# 02 - Log Parsing & Pattern Extraction

Notebook này gom log JSONL thành pattern/template để trả lời câu hỏi WHERE.

In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

BASE = Path(r'D:\\AWS\\AIOPS-study\\g2-data\\g2\\logs')
ART = Path(r'D:\\AWS\\AIOPS\\w1\\lab\\artifacts')
ART.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8')

In [ ]:
def normalize(msg: str) -> str:
    msg = re.sub(r'\b[0-9a-f]{16,}\b', '<hex>', msg, flags=re.I)
    msg = re.sub(r'\bORD-[A-Z0-9]+\b', 'ORD-<id>', msg)
    msg = re.sub(r'\b\d+\.\d+\b', '<num>', msg)
    msg = re.sub(r'\b\d+\b', '<num>', msg)
    return msg

def load_log(path: Path):
    rows = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            rec['template'] = normalize(rec['message'])
            rows.append(rec)
    return pd.DataFrame(rows)

cart = load_log(BASE / 'cart-service.log.jsonl')
order = load_log(BASE / 'order-service.log.jsonl')
cart.head()

In [ ]:
cart_counter = Counter(cart['template'])
order_counter = Counter(order['template'])
pd.DataFrame(cart_counter.most_common(20), columns=['template', 'count']).to_csv(ART / 'cart_templates_top20.csv', index=False)
pd.DataFrame(order_counter.most_common(20), columns=['template', 'count']).to_csv(ART / 'order_templates_top20.csv', index=False)
pd.DataFrame(cart_counter.most_common(10), columns=['template', 'count'])

In [ ]:
focus = cart[cart['template'].isin([
    'ProductCatalogCache eviction failed: heap pressure too high',
    'OutOfMemoryError imminent: available heap < <num>%',
    'Container OOMKilled: memory limit exceeded',
    'GC overhead limit warning: pause=<num>ms heap=<num>%',
])]
focus.groupby('template')['timestamp'].min().sort_values()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
top = pd.DataFrame(cart_counter.most_common(12), columns=['template', 'count'])
ax.barh(top['template'][::-1], top['count'][::-1])
ax.set_title('Top log templates in cart-service')
fig.tight_layout()
fig.savefig(ART / 'log_templates_top12.png', dpi=160, bbox_inches='tight')
plt.show()


Kết luận chính:
- Pattern `heap pressure too high` xuất hiện sớm.
- `GC overhead limit warning` là dấu hiệu chuyển sang trạng thái suy kiệt.
- `OutOfMemoryError imminent` và `OOMKilled` xác nhận failure mechanism.